# CA6003 — A: Data foundation and governance

研究问题：特殊值语义与未知信息处理如何影响银行营销分类的可靠性？

本章交付来源、字段政策、训练分区质量审计与固定划分。B 负责开放式发现，C 负责 S/U 四组预处理与 Notebook 总装，D 负责模型和评估。这里没有拟合模型或学习填补、缩放参数。

场景：历史记录层面的**联系前**离线名单筛选。缺少客户唯一 ID 和特征快照时间，不能证明客户级独立性或真实上线可用性；随机划分不代表未来泛化或因果营销效果。

## 1. Provenance, license and scope

仅使用 legacy `bank-full.csv`：45,211 行、16 个原始输入 + y。外 ZIP → bank.zip → 原始 CSV/说明。拒绝 bank-additional、bank.csv 抽样版和更改列名/分隔符的 data.csv。

来源：[UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)，许可：[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)。数据作者 Moro、Rita、Cortez；数据引用 DOI [10.24432/C5K306](https://doi.org/10.24432/C5K306)。旧版说明另要求引用 Moro、Laureano、Cortez (2011)，详见仓库 THIRD_PARTY_NOTICES.md。

原始字节不修改；row_id、划分及审计是本项目增加的派生成果。署名不表示作者或 UCI 背书。

在仓库根目录或 notebooks/ 下打开本章。所有路径相对项目，不下载个人资料。若尚未准备数据，先运行 `python scripts/prepare_data.py`。

In [1]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
ROOT = Path.cwd().resolve()
if not (ROOT / 'config/dataset_lock.json').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'config/dataset_lock.json').is_file(), 'Open from repository root or notebooks/'
sys.path.insert(0, str(ROOT))
lock = json.loads((ROOT / 'config/dataset_lock.json').read_text(encoding='utf-8'))
for relative, expected in lock['artifacts'].items():
    data = (ROOT / relative).read_bytes()
    assert hashlib.sha256(data).hexdigest() == expected['sha256'], relative
    assert 'bytes' not in expected or len(data) == expected['bytes'], relative
provenance = json.loads((ROOT / 'data/source/provenance.json').read_text(encoding='utf-8'))
print('Acquired UTC:', provenance['acquired_utc'])
print('Raw SHA-256:', lock['artifacts']['data/raw/bank-full.csv']['sha256'])
print('All four pinned source artifacts verified')

Acquired UTC: 2026-10-12T09:30:28.154035+00:00
Raw SHA-256: d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa264b3aaf0f849ed32d
All four pinned source artifacts verified


## 2. Read without semantic cleaning and freeze the exact split

`keep_default_na=False` 保留文字 unknown 和原空字符串；显式空值由审计单独检查。`row_id=np.arange(len(raw))` 对应原始数据行的 0-based 位置，不是客户标识，也不进入 X。两次 stratified split 严格使用 random_state=42；第二次必须使用 y.iloc[dev]。Manifest 按原始 row_id 排序，JSON 另保留 split 返回顺序。

In [2]:
raw = pd.read_csv(ROOT / 'data/raw/bank-full.csv', sep=';', keep_default_na=False)
assert list(raw.columns) == lock['columns'] and raw.shape == (45211, 17)
y = raw['y'].map({'no': 0, 'yes': 1})
assert not y.isna().any()
ids = np.arange(len(raw))
dev, test = train_test_split(ids, test_size=.20, stratify=y, random_state=42)
train, val = train_test_split(dev, test_size=.25, stratify=y.iloc[dev], random_state=42)
splits = {'train': train, 'val': val, 'test': test}
assert len(set(train) & set(val)) == 0
assert len(set(dev) & set(test)) == 0
assert set(np.concatenate(list(splits.values()))) == set(ids)
assignment = pd.Series('', index=ids, name='split')
for name, positions in splits.items():
    assignment.loc[positions] = name
manifest = assignment.rename_axis('row_id').reset_index()
pd.testing.assert_frame_equal(manifest, pd.read_csv(ROOT / 'data/splits/split_manifest.csv'))
saved = json.loads((ROOT / 'data/splits/split_indices.json').read_text(encoding='utf-8'))
assert all(np.array_equal(splits[name], saved[name]) for name in splits)
raw.index.name = 'row_id'
print('Shape:', raw.shape, 'split sizes:', {k: len(v) for k,v in splits.items()})

Shape: (45211, 17) split sizes: {'train': 27126, 'val': 9042, 'test': 9043}


以下表仅对 train 展示标签分布；val/test 只显示预定行数覆盖信息，其标签统计保持空白。开放 profiling 仅使用 train。封存是团队流程约束，原始 CSV 本身仍包含全部记录。

In [3]:
display(pd.read_csv(ROOT / 'outputs/split_summary.csv'))
format_checks = pd.read_csv(ROOT / 'outputs/format_checks.csv')
assert format_checks.n_violations.sum() == 0
print('Predeclared holdout format checks passed:', len(format_checks))

,partition,n_rows,n_no,n_yes,positive_rate,purpose
0,train,27126,23953.0,3173.0,0.116973,training profile
1,val,9042,NaN,NaN,NaN,structure/coverage only; label statistics with...
2,test,9043,NaN,NaN,NaN,structure/coverage only; label statistics with...


Predeclared holdout format checks passed: 78


## 3. Dictionary and prediction-time policy

11 个输入：age、job、marital、education、default、balance、housing、loan、pdays、previous、poutcome。

duration 是通话后信息；campaign、contact、day、month 也按保守的联系前时点排除。这是小组研究设计，而非 UCI 或课程强制限制。reviewer 保留 pending，待真实成员复核。

In [4]:
dictionary = pd.read_csv(ROOT / 'notes/data_dictionary.csv', keep_default_na=False)
policy = pd.read_csv(ROOT / 'notes/feature_policy.csv', keep_default_na=False)
assert dictionary.field.tolist() == lock['columns']
allowed = policy.loc[policy.role.eq('input'), 'field'].tolist()
assert allowed == lock['allowed_features']
assert not set(allowed) & {'y', 'row_id', 'duration', 'campaign', 'contact', 'day', 'month'}
display(dictionary)
display(policy[['field', 'role', 'prediction_time', 'reviewer']])

,field,storage_type,meaning,unit,documented_categories,special_semantics,role,source
0,age,integer,Client age,years,,Positive integer; no arbitrary age trimming,input,data/raw/bank-names.txt; UCI dataset 222
1,job,string,Type of job,category,admin.|unknown|unemployed|management|housemaid...,unknown is a literal category; U applies here,input,data/raw/bank-names.txt; UCI dataset 222
2,marital,string,Marital status,category,married|divorced|single,divorced includes divorced or widowed in the s...,input,data/raw/bank-names.txt; UCI dataset 222
3,education,string,Education category,category,unknown|secondary|primary|tertiary,Legacy primary/secondary/tertiary/unknown; U a...,input,data/raw/bank-names.txt; UCI dataset 222
4,default,string,Has credit in default?,yes/no,yes|no,Do not infer default amount or date,input,data/raw/bank-names.txt; UCI dataset 222
5,balance,integer,Average yearly balance,EUR,,Negative and extreme values are not automatica...,input,data/raw/bank-names.txt; UCI dataset 222
6,housing,string,Has housing loan?,yes/no,yes|no,Retain original categories,input,data/raw/bank-names.txt; UCI dataset 222
7,loan,string,Has personal loan?,yes/no,yes|no,Retain original categories,input,data/raw/bank-names.txt; UCI dataset 222
8,contact,string,Last contact communication type,category,unknown|telephone|cellular,Current campaign contact process; excluded,excluded,data/raw/bank-names.txt; UCI dataset 222
9,day,integer,Day of month of last contact,1-31,,Not day_of_week; incomplete calendar information,excluded,data/raw/bank-names.txt; UCI dataset 222


,field,role,prediction_time,reviewer
0,age,input,Assumed available before current campaign cont...,pending
1,job,input,Assumed available before current campaign cont...,pending
2,marital,input,Assumed available before current campaign cont...,pending
3,education,input,Assumed available before current campaign cont...,pending
4,default,input,Assumed available before current campaign cont...,pending
5,balance,input,Assumed available before current campaign cont...,pending
6,housing,input,Assumed available before current campaign cont...,pending
7,loan,input,Assumed available before current campaign cont...,pending
8,contact,excluded,Current/last-contact or campaign process; cons...,pending
9,day,excluded,Current/last-contact or campaign process; cons...,pending


## 4. Training-only quality evidence

本单元使用仓库内 `src/audit.py` 的审计函数（随本章一起提交，非个人外部脚本）。C 总装最终单一提交 Notebook 时，应将所需函数内联或按课程允许方式打包。

区分 NaN/空串、unknown、pdays=-1。pdays=-1 是此前未联系；pdays=0 是实际间隔。负余额、高联系次数、匹配记录不自动等于错误。STOP 级规则只有 n_affected>0 才阻塞；INVESTIGATE 表示待人工调查，不执行删除。

In [5]:
from src.audit import profile_train, audit_train
training = raw.loc[np.sort(train)].copy()
profile = profile_train(training)
audit = audit_train(training)
saved_audit = pd.read_csv(ROOT / 'outputs/quality_audit.csv', keep_default_na=False)
pd.testing.assert_frame_equal(audit, saved_audit)
assert audit.partition.eq('train').all()
assert audit.loc[audit.severity.eq('STOP'), 'n_affected'].sum() == 0
display(profile)
display(audit.loc[audit.n_affected.gt(0), ['check_id','field','n_affected','example_row_ids','severity','decision']])
print('No source rows dropped. No imputer/scaler/model fitted.')

,partition,field,dtype,n_rows,n_null,n_empty,n_whitespace_only,n_unknown,n_minus_one,n_unique,min,q01,q25,median,q75,q99,max
0,train,age,int64,27126,0,0,0,0,0,76,18.0,23.0,33.0,39.0,48.00,71.00,95.0
1,train,job,str,27126,0,0,0,179,0,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,train,marital,str,27126,0,0,0,0,0,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,train,education,str,27126,0,0,0,1100,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,train,default,str,27126,0,0,0,0,0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,train,balance,int64,27126,0,0,0,0,24,5896,-6847.0,-627.0,73.0,454.0,1425.75,13146.25,81204.0
6,train,housing,str,27126,0,0,0,0,0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,train,loan,str,27126,0,0,0,0,0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,train,contact,str,27126,0,0,0,7790,0,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,train,day,int64,27126,0,0,0,0,0,31,1.0,2.0,8.0,16.0,21.00,31.00,31.0


,check_id,field,n_affected,example_row_ids,severity,decision
3,unknown_job,job,179,"[4, 1072, 1292, 2046, 2841]",VALID_STATE,Retain raw category
8,unknown_education,education,1100,"[4, 13, 44, 57, 58]",VALID_STATE,Retain raw category
18,unknown_contact,contact,7790,"[0, 1, 4, 5, 6]",VALID_STATE,Retain raw category
28,unknown_poutcome,poutcome,22240,"[0, 1, 4, 5, 6]",VALID_STATE,Retain raw category
37,pdays_never,pdays,22235,"[0, 1, 4, 5, 6]",VALID_STATE,Retain sentinel in raw
39,negative_balance,balance,2280,"[25, 36, 37, 45, 46]",VALID_STATE,Retain negative balance
40,upper_tail_balance,balance,272,"[446, 2065, 2343, 2778, 2852]",INVESTIGATE,Retain; B reviews tail evidence
41,upper_tail_previous,previous,234,"[24136, 24295, 24331, 24556, 24596]",INVESTIGATE,Retain; B reviews tail evidence
44,history_contacted_unknown_outcome,"pdays,poutcome",5,"[40658, 41821, 42042, 43978, 45021]",INVESTIGATE,Retain unknown outcome
48,conflicting_labels_allowed11,"age,job,marital,education,default,balance,hous...",157,"[269, 378, 523, 787, 931]",INVESTIGATE,Retain; document identifiability limitation


No source rows dropped. No imputer/scaler/model fitted.


## 5. History consistency handoff

以下训练分区交叉计数用于检查语义与向 B 提供调查入口。观察是计数本身；不能据此补造联系史、解释 unknown 的真实生成原因或声称因果。B 仍须独立完成其核心发现、分层证据与限制。

In [6]:
history = pd.read_csv(ROOT / 'outputs/train_history_consistency.csv')
assert history.partition.eq('train').all() and history.n.sum() == len(training)
display(history)
summary = json.loads((ROOT / 'outputs/audit_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))

,partition,pdays_state,previous_state,poutcome,n,denominator
0,train,contacted_ge0,positive,failure,2876,27126
1,train,contacted_ge0,positive,other,1117,27126
2,train,contacted_ge0,positive,success,893,27126
3,train,contacted_ge0,positive,unknown,5,27126
4,train,never_-1,zero,unknown,22235,27126


{
  "raw_shape": [
    45211,
    17
  ],
  "split_sizes": {
    "train": 27126,
    "val": 9042,
    "test": 9043
  },
  "open_profile_partition": "train",
  "training_rows": 27126,
  "holdout_policy": "predeclared schema, category membership, numeric domain and row coverage checks only; no holdout label statistics",
  "format_violations": 0,
  "audit_stop_affected": 0,
  "training_exact_duplicate_group_members": 0,
  "training_duplicate_excess_rows": 0,
  "training_pdays_minus_one": 22235,
  "training_pdays_zero": 0,
  "training_unknown_job": 179,
  "training_unknown_education": 1100,
  "training_negative_balance": 2280,
  "rows_deleted": 0,
  "imputation_fitted": false,
  "scaler_fitted": false,
  "models_fitted": 0,
  "student_review": "pending",
  "C_semantic_review": "pending"
}


## 6. B / C / D interface and remaining reviews

- B：只用 train；保留 row_id 作证据定位，建立实际发现与分母。
- C：复用固定 X/y 和 11 列政策，实现四个 S/U 开关；只在 fit 分区学习填补、编码、缩放。U 只改变 job/education，poutcome 不变；S 区分 -1/0/正天数。
- D：相同原始行和划分、固定模型与阈值；规则冻结后重拟合 train+val，再对 test 一次性预定评估。

以下只加载 train/val 验证接口。D 最终阶段可显式 allow_test=True；这不是访问控制，也不代表现在可以查看 test 结果。

In [7]:
from src.data_contract import load_partition
X_train, y_train = load_partition('train')
X_val, y_val = load_partition('val')
assert X_train.index.equals(y_train.index)
assert X_train.columns.tolist() == allowed
assert np.array_equal(X_train.index, train)
print('Training:', X_train.shape, 'Validation:', X_val.shape)
try:
    load_partition('test')
except ValueError as exc:
    print(str(exc))

Training: (27126, 11) Validation: (9042, 11)
Test is sealed until protocol freeze; D must explicitly allow_test=True


## 7. GenAI declaration and review status

工具：OpenAI Codex（本任务 AI 编程辅助）。用途：起草 A 的下载/哈希校验、读取与固定拆分、审计、文档和 Notebook；根据本地运行结果修正。所有数值来自实际脚本，不采用手册示例作为已运行结果。实际验证见 outputs/validation.json 与 notes/QA.md：字节哈希、划分复算、训练审计复算、接口防泄漏检查、离线复跑、干净内核执行。

**学生人工复核：pending。C 的语义复核：pending。B/C/D 接收签字：pending。** 自动化检查不等于学生已理解并独立验证。成员须核对代码、字段时点与结论，填写真实姓名/贡献，保留 AI 使用声明；不把生成内容当作已完成的人工工作。

此 A 包不是完整课程提交；B 的发现、C 的四组管线/最终 Notebook、D 的模型结果及全组视频仍待完成。

参考：UCI 数据页与 data/raw/bank-names.txt；[scikit-learn 数据泄漏说明](https://scikit-learn.org/stable/common_pitfalls.html)；项目 notes/data_card.md、notes/DATA_CONTRACT.md、THIRD_PARTY_NOTICES.md。